In [18]:
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

import os
import cv2
import numpy as np
import time
from pynput.mouse import Button, Controller
os.environ['QT_QPA_PLATFORM'] = 'xcb'


In [19]:
base_options = python.BaseOptions(model_asset_path='/mnt/Main Drive/Codes/Deep Learning/Gesture_control/Mediapipe_Gesture/Hagrid_model_Initial/gesture_recognizer.task')
options = vision.GestureRecognizerOptions(base_options=base_options)
recognizer = vision.GestureRecognizer.create_from_options(options)

I0000 00:00:1731581142.765334    2154 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1731581142.768703   14478 gl_context.cc:357] GL version: 3.2 (OpenGL ES 3.2 Mesa 24.2.6-arch1.1), renderer: AMD Radeon Graphics (radeonsi, renoir, LLVM 18.1.8, DRM 3.54, 6.6.59-1-lts)
W0000 00:00:1731581142.769571    2154 gesture_recognizer_graph.cc:129] Hand Gesture Recognizer contains CPU only ops. Sets HandGestureRecognizerGraph acceleration to Xnnpack.
W0000 00:00:1731581142.798687   14479 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1731581142.825593   14489 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1731581142.827912   14487 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling supp

**Functions**

In [20]:
def Get_Gesture(frame):

    image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame)
    recognition_result = recognizer.recognize(image)

    if recognition_result.gestures:
        top_gesture = recognition_result.gestures[0][0].category_name
        # print(top_gesture)
    else:
        top_gesture = "Nothing"
        
    return top_gesture

In [21]:
def Image_Processing(frame,hands,top_gesture):

    image = cv2.cvtColor(cv2.flip(frame,1),cv2.COLOR_BGR2RGB)
    image.flags.writeable = False

    results = hands.process(image)
    
    image.flags.writeable = True
    image = cv2.cvtColor(image,cv2.COLOR_RGB2BGR)

    fontScale = 2
    fontFace = cv2.FONT_HERSHEY_PLAIN
    fontColor = (0,255,0)
    fontThickness = 2

    cv2.putText(image,top_gesture,(0,30),fontFace,fontScale,fontColor,fontThickness,cv2.LINE_AA)


    return image,results

In [22]:
class Gesture_Action:

    def __init__(self):
        self.mouse = Controller()
        self.w = 640
        self.h = 480
        self.x = 0  # Initialize previous x coordinate
        self.y = 0  # Initialize previous y coordinate

    def Move_Cursor(self, results):
        if results.multi_hand_landmarks:
            hand_landmarks = results.multi_hand_landmarks[0]  # Access the first hand
            current_x = hand_landmarks.landmark[8].x * self.w  # Scale current x coordinate
            current_y = hand_landmarks.landmark[8].y * self.h  # Scale current y coordinate
            cx, cy = current_x - self.x, current_y - self.y  # Calculate difference
            self.mouse.move(cx * 5, cy *5)  # Move cursor by the difference
            self.x, self.y = current_x, current_y  # Update previous coordinates


    def Gesture_Action(self,top_gesture,results):
        mouse = self.mouse
        
        if top_gesture == "one" or top_gesture == "mute":
            self.Move_Cursor(results)
        elif top_gesture == "ok":
            mouse.press(Button.left)
        elif top_gesture == "Close_L_shape":
            pass
        elif top_gesture == "L_shape":
            pass
        elif top_gesture == "Close_Palm":
            pass
        elif top_gesture == "Open_Palm":
            pass
        elif top_gesture == "scissors":
            pass

**Mouse**

In [23]:
mp_drawing = mp.solutions.drawing_utils
mp_hands = mp.solutions.hands
s = "http://192.168.195.103:4747/video"
s1 = 0
s2 = 1
cap = cv2.VideoCapture(s2)

G = Gesture_Action()


Xlib.xauth: warning, no xauthority details available


In [24]:
with mp_hands.Hands(static_image_mode=False, max_num_hands=1, min_detection_confidence=0.7, min_tracking_confidence=0.5) as hands:
    while cap.isOpened():
        ret,frame = cap.read()
        
        if not ret:
            print("Ignoring empty camera frame.")
            break

        top_gesture = Get_Gesture(frame)

        image,results = Image_Processing(frame,hands,top_gesture)
        
        h,w,c = image.shape

        if results.multi_hand_landmarks:
            for hand_landmarks in results.multi_hand_landmarks:

                cx, cy = hand_landmarks.landmark[8].x, hand_landmarks.landmark[8].y

                hand_landmarks,mp_drawing.draw_landmarks(image, hand_landmarks, mp_hands.HAND_CONNECTIONS)
               
            G.Gesture_Action(top_gesture,results)
            cv2.imshow('Hand Tracking', image)

        else:
            cv2.imshow('Hand Tracking', image)

        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

I0000 00:00:1731581143.010537    2154 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1731581143.014127   14515 gl_context.cc:357] GL version: 3.2 (OpenGL ES 3.2 Mesa 24.2.6-arch1.1), renderer: AMD Radeon Graphics (radeonsi, renoir, LLVM 18.1.8, DRM 3.54, 6.6.59-1-lts)
W0000 00:00:1731581143.048063   14496 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1731581143.073247   14509 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Prototype Code


In [25]:
# with mp_hands.Hands(static_image_mode=False, max_num_hands=2, min_detection_confidence=0.7, min_tracking_confidence=0.5) as hands:
#     while cap.isOpened():
#         ret,frame = cap.read()
        
#         if not ret:
#             print("Ignoring empty camera frame.")
#             break

#         # image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame)
#         # recognition_result = recognizer.recognize(image)
#         # if recognition_result.gestures:
#         #     top_gesture = recognition_result.gestures[0][0].category_name
#         #     # print(top_gesture)

#         # else:
#         #     top_gesture = "Nothing"
#         top_gesture = Get_Gesture(frame)


#         # image = cv2.cvtColor(cv2.flip(frame,1),cv2.COLOR_BGR2RGB)

#         # image.flags.writeable = False

#         # results = hands.process(image)
#         # image.flags.writeable = True

#         # image = cv2.cvtColor(image,cv2.COLOR_RGB2BGR)
#         image,results = Image_Processing(frame,hands,top_gesture)
#         # fontScale = 2
#         # fontFace = cv2.FONT_HERSHEY_PLAIN
#         # fontColor = (0,255,0)
#         # fontThickness = 2

#         # Draw bounding box
#         h,w,c = image.shape


#         if results.multi_hand_landmarks:
#             for hand_landmarks in results.multi_hand_landmarks:
#                 cx, cy = int(hand_landmarks.landmark[8].x * w), int(hand_landmarks.landmark[8].y * h)

#                 hand_landmarks,mp_drawing.draw_landmarks(image, hand_landmarks, mp_hands.HAND_CONNECTIONS)
               
        
 
    

#             # check = GEST_V(image,hand_landmarks.landmark[8],hand_landmarks.landmark[5],hand_landmarks.landmark[12],hand_landmarks.landmark[9])
#             # print (check)
#             # if check == 1:
#             screen_cx, screen_cy = calibrate_coordinates_sensitivity(cx, cy)
#             mouse.position = (screen_cx, screen_cy)

#             cv2.imshow('Hand Tracking', image)
#             # pyautogui.moveTo(screen_cx, screen_cy)

#         else:
#             cv2.imshow('Hand Tracking', image)

#         if cv2.waitKey(10) & 0xFF == ord('q'):
#             break

# cap.release()
# cv2.destroyAllWindows()